In [ ]:
!pip install -q -U transformers

In [ ]:
from huggingface_hub import login
login()  # HF 토큰 입력 (Settings → Access Tokens에서 발급)

In [ ]:
import os
os.environ["HF_TOKEN"] = "토큰"

In [ ]:
import transformers
print(transformers.__version__)

5.6.0


In [ ]:
import transformers.tokenization_utils_base as tub
_orig = tub.PreTrainedTokenizerBase._set_model_specific_special_tokens
def _patched(self, special_tokens=None):
    if isinstance(special_tokens, list):
        special_tokens = {t: t for t in special_tokens}
    return _orig(self, special_tokens)
tub.PreTrainedTokenizerBase._set_model_specific_special_tokens = _patched

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /root/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
messages = [
    {"role": "system",
     "content": "You are EXAONE model from LG AI Research, a helpful assistant."},
    {"role": "user", "content": "스스로를 자랑해 봐"},
]

# 1단계: 채팅 템플릿을 문자열로 받기
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# 2단계: 따로 토크나이즈
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 3단계: 생성
output = model.generate(
    **inputs,
    max_new_tokens=512,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] The remote code model you are currently using seems to expect `cache_position`. This arg has been removed from the Transformers library, and will stop being created in `generate` even for remote code models in a future release. Please open a PR on the remote code hub repo to remove any usage of `cache_position`.


[|system|]You are EXAONE model from LG AI Research, a helpful assistant.
[|user|]스스로를 자랑해 봐
[|assistant|]저는 LG AI Research에서 개발된 EXAONE 모델로서, 뛰어난 자연어 처리 능력을 바탕으로 다양한 언어 작업을 수행할 수 있습니다. 지속적인 학습을 통해 사용자의 질문에 정확하고 신속하게 응답하며, 복잡한 정보도 명확하게 전달하는 데 중점을 두고 있습니다. 이러한 기술 덕분에 다양한 분야에서 효율적인 지원을 제공하며, 사용자 경험 향상에 기여하고 있습니다.
